# Train LNL-Ti on GTSRB → plug-and-play `.pth` (target Top-1 ≥ 99.5%)

This notebook **produces the weights we ship**. The teacher will load that `.pth` into our
`LNL.py` and run the **original Test cell** (raw `[0,1]` images, plain `Linear(192,43)` head, no TTA).
So this notebook is built to match that contract **exactly**:

* dataloaders feed **raw `[0,1]`** images (Resize+ToTensor only) — identical to the grader's Test cell;
* normalization + train-time augmentation live **inside `LNL.py`** (`forward_features`), so train-time
  and verify-time preprocessing are the *same code path* (no mismatch possible);
* the classification head is a **plain `Linear(192,43)`** — exactly what the grader builds;
* the strong recipe (AdamW, cosine+warmup, label smoothing, EMA, ~30 epochs) is **ours**, used only to
  produce the weights. The shipped file + weights + Test cell are what must line up — and they do.

**How to run:** Runtime → Change runtime type → **GPU (T4)** → then **Runtime → Run all**.
It checkpoints to Google Drive every epoch and **auto-resumes** after a disconnect — just re-run all.

⏱️ ~30 epochs on a free T4 ≈ 2–3 h. If it disconnects, reopen and Run all; it continues where it stopped.

## 0. Mount Google Drive (checkpoint + resume)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
CKPT_DIR = '/content/drive/MyDrive/lnl_gtsrb'
os.makedirs(CKPT_DIR, exist_ok=True)
CKPT      = os.path.join(CKPT_DIR, 'last.pth')      # full training state (resume)
BEST_PTH  = os.path.join(CKPT_DIR, 'lnl_gtsrb.pth') # PLUG-AND-PLAY weights to ship
print('checkpoints ->', CKPT)
print('ship weights ->', BEST_PTH)

## 1. GPU check + clone our repo + install deps

In [ ]:
!nvidia-smi -L
%cd /content
!rm -rf LNL-GTSRB
!git clone -q https://github.com/bnqtoan/LNL-GTSRB.git
%cd /content/LNL-GTSRB
!pip install -q timm einops 2>&1 | tail -1
print('repo + deps ready')

Sanity-check that the **improved** `LNL.py` (in-model norm + aug) is the one we'll train. These flags must both be `True`.

In [ ]:
import sys, importlib
sys.path.insert(0, '/content/LNL-GTSRB')
import LNL as LNLmod; importlib.reload(LNLmod)
_m = LNLmod.LNL_Ti(pretrained=False)
print('in_model_norm:', _m.in_model_norm, '| in_model_aug:', _m.in_model_aug)
print('has LayerScale:', any('ls_attn_in' in n for n,_ in _m.named_parameters()))
del _m

## 2. Download + prepare GTSRB
Same dataset and same test-folder layout as the original `Instructions.ipynb`.

In [ ]:
import os, shutil
!mkdir -p data
base='https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370'
if not os.path.exists('data/GTSRB/Final_Training/Images'):
    !curl -sL $base/GTSRB_Final_Training_Images.zip -o data/train.zip
    !curl -sL $base/GTSRB_Final_Test_Images.zip     -o data/test.zip
    !curl -sL $base/GTSRB_Final_Test_GT.zip         -o data/test_gt.zip
    !unzip -q -o data/train.zip   -d data/
    !unzip -q -o data/test.zip    -d data/
    !unzip -q -o data/test_gt.zip -d data/
data_dir='./data/GTSRB'; images_dir=os.path.join(data_dir,'Final_Test/Images')
test_dir=os.path.join(data_dir,'test'); os.makedirs(test_dir, exist_ok=True)
if len(os.listdir(test_dir)) < 43:
    with open('./data/GT-final_test.csv') as f: rows=f.readlines()
    for t in rows[1:]:
        cls=int(t.split(';')[-1]); name=t.split(';')[0]
        d=os.path.join(test_dir, f'{cls:04d}'); os.makedirs(d, exist_ok=True)
        shutil.copy(os.path.join(images_dir, name), d)
print('GTSRB ready | test classes:', len(os.listdir(test_dir)))

## 3. Dataloaders — **RAW `[0,1]`, exactly like the grader**
Only `Resize((224,224))` + `ToTensor()`. **No** normalization and **no** augmentation here —
those live inside `LNL.py` and fire automatically (`forward_features`; aug only when `model.training`).
This is the key to plug-and-play: the model sees the *same* raw input at train time and at the
teacher's verify time. We also carve a small **validation split** off the training set so we never
select the model on the test set.

In [ ]:
import torch, torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, random_split

IMG = 224
BATCH = 64            # T4-friendly; raw 224 images
raw_tf = T.Compose([T.Resize((IMG, IMG)), T.ToTensor()])   # RAW [0,1] — no norm, no aug

full_train = torchvision.datasets.ImageFolder('./data/GTSRB/Final_Training/Images', transform=raw_tf)
testset    = torchvision.datasets.ImageFolder('./data/GTSRB/test',                  transform=raw_tf)

# deterministic 97/3 train/val split for model selection
n_val   = max(1, int(0.03 * len(full_train)))
n_train = len(full_train) - n_val
g = torch.Generator().manual_seed(42)
trainset, valset = random_split(full_train, [n_train, n_val], generator=g)

train_loader = DataLoader(trainset, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(valset,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(testset,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print(f'train {len(trainset)} | val {len(valset)} | test {len(testset)} | classes {len(full_train.classes)}')
assert len(full_train.classes) == 43, 'expected 43 GTSRB classes'

## 4. Build the model — **exactly as the grader will**
`LNL_Ti(pretrained=False)` then `model.head = Linear(192, 43)`. Training with this exact head means
the saved `state_dict` keys match the grader's model 1:1 → `load_state_dict(strict=True)` is clean.

In [ ]:
import torch.nn as nn
model = LNLmod.LNL_Ti(pretrained=False)
model.head = nn.Linear(in_features=192, out_features=43, bias=True)   # exact grader head
model = model.cuda()
n_param = sum(p.numel() for p in model.parameters())/1e6
print(f'LNL-Ti ready | {n_param:.2f}M params | head = {model.head}')

## 5. Train (AdamW + cosine+warmup + label smoothing + EMA) with **checkpoint-resume**
The recipe is ours and only produces the weights. EMA keeps a smoothed copy of the weights that
usually generalizes better. Every epoch we save full state to Drive (`last.pth`) and, whenever the
**validation** accuracy improves, we save the plug-and-play `lnl_gtsrb.pth`. Re-running the notebook
after a disconnect resumes from `last.pth` automatically.

In [ ]:
import math, time, copy

EPOCHS   = 30
WARMUP   = 3
BASE_LR  = 6e-4
WD       = 0.05
EMA_DECAY = 0.9995

loss_fn   = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WD)
scaler    = torch.amp.GradScaler('cuda')

def lr_at(epoch):
    if epoch < WARMUP:
        return BASE_LR * (epoch + 1) / WARMUP
    p = (epoch - WARMUP) / max(1, (EPOCHS - WARMUP))
    return 0.5 * BASE_LR * (1 + math.cos(math.pi * p))

# EMA shadow model
ema = copy.deepcopy(model).eval()
for p in ema.parameters(): p.requires_grad_(False)
@torch.no_grad()
def ema_update():
    for e, m in zip(ema.parameters(), model.parameters()):
        e.mul_(EMA_DECAY).add_(m.detach(), alpha=1 - EMA_DECAY)
    for e, m in zip(ema.buffers(), model.buffers()):
        e.copy_(m)

@torch.no_grad()
def accuracy(net, loader):
    net.eval(); correct = total = 0
    for x, y in loader:
        x = x.cuda(non_blocking=True)
        with torch.amp.autocast('cuda'):
            out = net(x)
        correct += (out.argmax(1) == y.cuda(non_blocking=True)).sum().item()
        total   += y.size(0)
    return 100.0 * correct / total

# ---- resume ----
start_epoch = 0; best_val = 0.0
if os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location='cuda')
    model.load_state_dict(ck['model']); ema.load_state_dict(ck['ema'])
    optimizer.load_state_dict(ck['optim']); scaler.load_state_dict(ck['scaler'])
    start_epoch = ck['epoch'] + 1; best_val = ck.get('best_val', 0.0)
    print(f'resumed at epoch {start_epoch}/{EPOCHS} | best_val so far {best_val:.2f}%')
else:
    print('training from scratch')

t0 = time.time()
for epoch in range(start_epoch, EPOCHS):
    lr = lr_at(epoch)
    for g_ in optimizer.param_groups: g_['lr'] = lr
    model.train(); running = 0.0
    for x, y in train_loader:
        x = x.cuda(non_blocking=True); y = y.cuda(non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            out = model(x); cost = loss_fn(out, y)
        scaler.scale(cost).backward()
        scaler.step(optimizer); scaler.update()
        ema_update(); running += cost.item()
    # select on EMA (usually best) but track both
    val_ema = accuracy(ema, val_loader)
    val_raw = accuracy(model, val_loader)
    val_best = max(val_ema, val_raw)
    pick = 'ema' if val_ema >= val_raw else 'raw'
    print(f'epoch {epoch+1:2d}/{EPOCHS} | lr {lr:.2e} | loss {running/len(train_loader):.4f} '
          f'| val_raw {val_raw:.2f} | val_ema {val_ema:.2f} | ({(time.time()-t0)/60:.1f} min)')
    # always save resume state
    torch.save({'model':model.state_dict(),'ema':ema.state_dict(),'optim':optimizer.state_dict(),
                'scaler':scaler.state_dict(),'epoch':epoch,'best_val':best_val}, CKPT)
    # save ship weights when val improves
    if val_best > best_val:
        best_val = val_best
        ship = ema if pick == 'ema' else model
        torch.save(ship.state_dict(), BEST_PTH)
        # keep best_val in resume file too
        torch.save({'model':model.state_dict(),'ema':ema.state_dict(),'optim':optimizer.state_dict(),
                    'scaler':scaler.state_dict(),'epoch':epoch,'best_val':best_val}, CKPT)
        print(f'    ↑ new best val {best_val:.2f}% ({pick}) → saved {BEST_PTH}')
print(f'training complete | best val {best_val:.2f}%')

## 6. ✅ Plug-and-play self-verification (this is the number that matters)
We now **discard the in-memory model** and rebuild it from scratch *exactly as the grader does*,
load `lnl_gtsrb.pth` with `strict=True`, set `eval()`, and run the **original Test loop on raw `[0,1]`**
(no TTA, no normalization in the loader). If this prints **≥ 99.5%**, the shipped weights are
guaranteed plug-and-play.

In [ ]:
import importlib, torch, torch.nn as nn
import LNL as LNLmod; importlib.reload(LNLmod)   # fresh module, like a fresh notebook

fresh = LNLmod.LNL_Ti(pretrained=False)
fresh.head = nn.Linear(in_features=192, out_features=43, bias=True)
missing, unexpected = fresh.load_state_dict(torch.load(BEST_PTH, map_location='cpu'), strict=False)
print('missing keys :', missing)
print('unexpected   :', unexpected)
assert not missing and not unexpected, 'STATE_DICT MISMATCH — weights are NOT plug-and-play!'
fresh = fresh.cuda().eval()

# --- original Test cell, verbatim semantics ---
correct = 0; total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images = images.cuda()
        outputs = fresh(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels.cuda()).sum().item()
verify_acc = 100 * float(correct) / total          # captured for the auto-upload gate below
print('Standard accuracy: %.2f %%' % verify_acc)
print('-> THIS is the number the teacher will reproduce. Must be >= 99.50.')

## 7. Fingerprint the shippable weights
`lnl_gtsrb.pth` is on Drive (backup). Print size + sha256 so the exact file is identifiable later.

In [ ]:
import hashlib, os
sz = os.path.getsize(BEST_PTH)/1e6
sha = hashlib.sha256(open(BEST_PTH,'rb').read()).hexdigest()
print(f'{BEST_PTH} | {sz:.1f} MB | sha256:{sha}')

## 8. ⬆️ Upload `lnl_gtsrb.pth` to the GitHub Release — **directly from Colab**
Uploads to release **`weights-v1`** on **`bnqtoan/LNL-GTSRB`** (asset `lnl_gtsrb.pth`), overwriting any existing asset.
After this, `Instructions_NhomX.ipynb` will auto-download these exact weights — **nothing to do locally.**

You'll be asked for a **GitHub token** (typed via `getpass`, never stored in the notebook). Use either:
* a **fine-grained PAT** scoped to repo `bnqtoan/LNL-GTSRB` with **Contents: Read and write**, or
* a **classic PAT** with the **`repo`** scope.

Create one at <https://github.com/settings/tokens>. The upload is **gated on the section-6 accuracy**:
it refuses to publish weights below 99.50% unless you explicitly override.

In [ ]:
import os, json, requests, hashlib
from getpass import getpass

REPO = 'bnqtoan/LNL-GTSRB'
TAG  = 'weights-v1'
ASSET_NAME = 'lnl_gtsrb.pth'
ALLOW_BELOW_THRESHOLD = False   # set True only if you deliberately want to ship < 99.50%

# --- gate on the self-verify accuracy from section 6 ---
try:
    _acc = float(verify_acc)
except NameError:
    raise RuntimeError('Run section 6 (self-verification) first — verify_acc is not defined.')
if _acc < 99.5 and not ALLOW_BELOW_THRESHOLD:
    raise RuntimeError(f'Self-verify accuracy {_acc:.2f}% < 99.50%. Not uploading. '
                       f'Train more, or set ALLOW_BELOW_THRESHOLD=True to override.')
print(f'gate OK — accuracy {_acc:.2f}%; preparing upload of {ASSET_NAME}')

TOKEN = getpass('GitHub token (fine-grained Contents:RW, or classic repo scope): ').strip()
H = {'Authorization': f'Bearer {TOKEN}', 'Accept': 'application/vnd.github+json',
     'X-GitHub-Api-Version': '2022-11-28'}
API = 'https://api.github.com'

# 1) find the release by tag
r = requests.get(f'{API}/repos/{REPO}/releases/tags/{TAG}', headers=H)
if r.status_code == 404:
    raise RuntimeError(f'Release tag {TAG} not found on {REPO}. Create it first (the assistant did this).')
r.raise_for_status()
rel = r.json(); rel_id = rel['id']
print(f'release {TAG} id={rel_id}')

# 2) delete an existing asset of the same name (overwrite)
for a in rel.get('assets', []):
    if a['name'] == ASSET_NAME:
        d = requests.delete(f'{API}/repos/{REPO}/releases/assets/{a["id"]}', headers=H)
        d.raise_for_status(); print(f'deleted existing asset {ASSET_NAME} (id={a["id"]})')

# 3) upload the new asset
up = f'https://uploads.github.com/repos/{REPO}/releases/{rel_id}/assets?name={ASSET_NAME}'
data = open(BEST_PTH, 'rb').read()
uh = dict(H); uh['Content-Type'] = 'application/octet-stream'
print(f'uploading {len(data)/1e6:.1f} MB ...')
u = requests.post(up, headers=uh, data=data)
u.raise_for_status()
asset = u.json()
print('uploaded:', asset['browser_download_url'])
print('size on release: %.1f MB' % (asset['size']/1e6))
print('local sha256   :', hashlib.sha256(data).hexdigest())
expected = f'https://github.com/{REPO}/releases/download/{TAG}/{ASSET_NAME}'
print('verify-notebook expects:', expected)
assert asset['browser_download_url'] == expected, 'URL mismatch — check tag/asset name'
print('\n✅ Done. Now just run Instructions_NhomX.ipynb (Run all) to confirm the teacher path.')

### Report back to the human
1. The **`Standard accuracy`** from **section 6** (must be ≥ 99.50).
2. The **`uploaded:`** URL from **section 8** (confirms the weights are live on the Release).

That's it — the whole loop ran in Colab; no local steps.